# 02 — Silver: Product Catalogue (M3)

**Owner:** M3 • **Tier:** Light • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)
**Hard deadline:** Day 5 EOD (Sat 16 May) — your `silver_product_master` is read by M5, M6, Lead.

## Scope — simplified per new allocation
- **Source tables (5):** `pc_products`, `pc_brands`, `pc_categories`, `pc_condition_codes`, `pc_price_history` (empty schema)
- **Silver transform owned: T5 — Product master, EXACT SKU pass only.** Fuzzy second-pass is deferred to lead/M2 (they'll handle the fuzzy match against marketplace listings during Phase 5/6).
- **Anomalies in your territory:** A9, B4 (2)

## Hint script (no counts revealed)
- **A9 (SKU mapped to different products in catalogue vs marketplace):** profile every distinct SKU in `ts_seller_listings`. For each, look up its catalogue product_name. If a seller's `product_name` is wildly different from catalogue's (e.g. one says "laptop", other says "phone case"), flag `SKU_PRODUCT_MISMATCH`.
- **B4 (NL free-text matching):** fuzzy-match `nl_listings.product_title` to `pc_products.product_name`. Defend your confidence threshold (≥0.85 reasonable). Document in report Section 7.

## Hands-on-of-everything checklist
- [ ] Bronze read (you'll read pc_* + sample ts_seller_listings)
- [ ] Silver write: 5 tables + product_master
- [ ] Gold dim: dim_product (SCD2 — see notebooks/03_gold_dimensions.ipynb), dim_listing_condition (static)
- [ ] Anomalies: A9 + B4
- [ ] Bus Matrix: justify cells where dim_product joins to facts (most of them)
- [ ] Report: contribute Sections 1, 2, 7 (T5), 8 (A9, B4), 9 (your dims)

> **Note:** you don't own a fact directly. Your compensation is the most-shared SCD2 dim and the most Bus Matrix cell justifications.


## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Cell 3 — Lookup Silver tables (small, easy)

**TODO:** for each of `pc_brands`, `pc_categories`, `pc_condition_codes`:
- Read from Bronze
- Generate SK on the natural key
- Add 4 mandatory anomaly columns (default CLEAN/CONFIRMED)
- Write to Silver

In [ ]:
# TODO M3:
# brands = read_bronze('pc_brands').withColumn('brand_key', surrogate_key(F.col('brand_id')))
# brands = add_anomaly_columns(brands)
# write_silver(brands, 'silver_brands')
#
# Repeat for pc_categories → silver_categories (key on category_id)
# Repeat for pc_condition_codes → silver_condition_codes (key on condition_code)

## Cell 4 — T5 Pass 1: exact SKU match (catalogue ↔ marketplace)

**TODO:**
- Read `pc_products` (catalogue) and `ts_seller_listings` (marketplace)
- Inner-join on `sku`
- For matched rows: `match_confidence = 1.00`, `canonical_product_key = surrogate_key(catalogue.sku)`
- For seller-only rows (SKU not in catalogue): flag `SKU_NOT_IN_CATALOGUE`
- For each matched row, compare seller's `product_name` to catalogue's. If disagreement is strong (use Levenshtein distance > N or rapidfuzz ratio < 50), flag `SKU_PRODUCT_MISMATCH` — this is anomaly A9

In [ ]:
# TODO M3: implement Pass 1
# catalogue = read_bronze('pc_products')
# seller_listings = read_bronze('ts_seller_listings')
#
# pass1 = (catalogue.alias('c').join(seller_listings.alias('s'),
#          F.col('c.sku') == F.col('s.sku'), 'left'))
# # ... add match_confidence, canonical_product_key, anomaly columns
# # ... flag mismatches

## Cell 5 — T5 Pass 2: fuzzy on (category + brand + model_name)

For seller listings unmatched by Pass 1, attempt a fuzzy match within the same category. Use `rapidfuzz.fuzz.token_sort_ratio` on `(brand + model_name)`.

Threshold ≥ 70 → match with `match_confidence = ratio/100` (0.70-0.99 band). Flag with `PRODUCT_FUZZY_MATCH`, status `FLAGGED_AMBIGUOUS`.

In [ ]:
# TODO M3: implement Pass 2 with rapidfuzz UDF
# from rapidfuzz import fuzz
# from pyspark.sql.types import IntegerType
#
# fuzz_ratio = F.udf(lambda a, b: fuzz.token_sort_ratio(a or '', b or ''), IntegerType())
# # Apply within same category as a blocking key

## Cell 6 — Build `silver_product_master`

Union Pass 1 matched + Pass 2 matched + unmatched (low confidence). Output one canonical row per `canonical_product_key` carrying:
- canonical_product_key (SK)
- sku
- product_name (catalogue value wins)
- brand_key, category_key, condition_code_key (FKs to lookup Silver tables)
- list_price, cogs (from catalogue)
- match_confidence
- 4 mandatory anomaly columns

In [ ]:
# TODO M3: union and write
# product_master = pass1.unionByName(pass2, allowMissingColumns=True)
# product_master = add_anomaly_columns(product_master)
# # apply flag() calls per Cell 4 + Cell 5 detection rules
# write_silver(product_master, 'silver_product_master')

## Cell 7 — Acceptance check

Before requesting PR review:
1. `silver_product_master` has unique `canonical_product_key` (no duplicates)
2. `match_confidence` ∈ [0.0, 1.00] for every row
3. Every row has all 4 mandatory anomaly columns populated
4. At least one row has `SKU_PRODUCT_MISMATCH` flagged (A9 — there is one in the data)
5. Catalogue's product_name appears (not seller's) on matched rows where they disagreed

In [ ]:
# pm = spark.read.format('snowflake').options(**sf_silver).option('dbtable', 'silver_product_master').load()
# pm.groupBy('match_confidence').count().orderBy('match_confidence').show()
# pm.groupBy('anomaly_reason_code').count().show(truncate=False)
# print('Distinct canonical keys:', pm.select('canonical_product_key').distinct().count())
# print('Total rows:', pm.count())